# Challenge 9 - Análisis de Actividad de Producto
### 1️⃣ Exploración Inicial (Medir antes de limpiar)

In [ ]:
import pandas as pd

# Cargar dataset
df_raw = pd.read_csv('docs/product_activity.csv')

# Mostrar primeros registros, info y descriptivas
display(df_raw.head())
df_raw.info()
display(df_raw.describe())

# Contar nulos y duplicados exactos
print("Nulos por columna:\n", df_raw.isnull().sum())
print("\nDuplicados exactos:", df_raw.duplicated().sum())

# Valores unicos y frecuencias
for col in ['plan_type', 'post_category', 'device_type']:
    print(f"\nFrecuencias de {col}:\n", df_raw[col].value_counts(dropna=False))

# Chequear logica de posts antes del registro
dt_signup = pd.to_datetime(df_raw['created_at'], errors='coerce')
dt_post = pd.to_datetime(df_raw['post_created_at'], errors='coerce')
invalid_posts_count = (dt_post < dt_signup).sum()
print(f"\nPosts antes del signup: {invalid_posts_count}")

### 2️⃣ Limpieza Básica y Data Quality Report

In [ ]:
df_clean = df_raw.copy()

# Diccionarios de mapeo para normatizar
plan_map = {'free': 'free', 'pro': 'pro', 'enterprise': 'enterprise'}
device_map = {'web': 'web', 'mobile': 'mobile', 'desktop': 'desktop'}
category_map = {
    'tech': 'tech', 'life': 'life', 'sports': 'sports', 'science': 'science',
    'finance': 'finance', 'gaming': 'gaming', 'music': 'music',
    'health': 'health', 'education': 'education', 'travel': 'travel'
}

# Funcion que limpia los valores segun los mapas
def clean_col_value(val, mapping):
    if pd.isna(val): return 'invalid'
    clean_val = str(val).lower().strip()
    return mapping.get(clean_val, 'invalid')

# Aplicar limpieza a categorias
df_clean['plan_type'] = df_clean['plan_type'].apply(lambda x: clean_col_value(x, plan_map))
df_clean['device_type'] = df_clean['device_type'].apply(lambda x: clean_col_value(x, device_map))
df_clean['post_category'] = df_clean['post_category'].apply(lambda x: clean_col_value(x, category_map))

# Parsear fechas y recalcular dias transcurridos
df_clean['created_at'] = pd.to_datetime(df_clean['created_at'], errors='coerce')
df_clean['post_created_at'] = pd.to_datetime(df_clean['post_created_at'], errors='coerce')
df_clean['days_since_signup_calc'] = (df_clean['post_created_at'] - df_clean['created_at']).dt.days

# Mascaras de cuarentena para buscar errores fuertes
mask_date_err = df_clean['created_at'].isna() | df_clean['post_created_at'].isna()
mask_logic_err = df_clean['days_since_signup_calc'] < 0
mask_dict_err = (df_clean['plan_type'] == 'invalid') | (df_clean['device_type'] == 'invalid') | (df_clean['post_category'] == 'invalid')

# Filtrar y separar validos e invalidos
mask_quarantine = mask_date_err | mask_logic_err | mask_dict_err
df_quarantine = df_clean[mask_quarantine].copy()
df_core = df_clean[~mask_quarantine].copy().drop_duplicates()

# Determinar motivo del fallo
def get_failure_reason(row):
    if pd.isna(row['created_at']) or pd.isna(row['post_created_at']): return 'date_error'
    if row['days_since_signup_calc'] < 0: return 'post_before_signup'
    return 'invalid_col'

if not df_quarantine.empty:
    df_quarantine['reason_code'] = df_quarantine.apply(get_failure_reason, axis=1)

# Reporte final de calidad
total_raw = len(df_raw)
total_core = len(df_core)
quarantine_pct = (len(df_quarantine) / total_raw) * 100
dupes_removed = len(df_clean[~mask_quarantine]) - total_core
mismatches_count = (df_core['days_since_signup'] != df_core['days_since_signup_calc']).sum()
mismatch_pct = (mismatches_count / total_core) * 100

print(f"RAW: {total_raw} filas | CORE (limpias): {total_core} filas")
print(f"Quarentena: {quarantine_pct:.2f}% | Duplicados eliminados: {dupes_removed}")
print(f"Mismatch en calculo de fechas: {mismatch_pct:.2f}%")

### 3️⃣ Métricas, Segmentación y Exportación

In [ ]:
# Distribuciones por plan, pais, dispositivo y categoria
print("Usuarios unicos por plan:\n", df_core.groupby('plan_type')['user_id'].nunique())
print("\nPosts por pais (Top 5):\n", df_core['country'].value_counts().head())
print("\nPosts por dispositivo:\n", df_core['device_type'].value_counts())
print("\nPosts por categoria:\n", df_core['post_category'].value_counts())

# Engagement general
print("\nVotos por plan (media y mediana):\n", df_core.groupby('plan_type')['votes_received'].agg(['mean', 'median']))

avg_votes_post = df_core['votes_received'].mean()
avg_votes_user = df_core.groupby('user_id')['votes_received'].mean().mean()
print(f"\nVotos promedio por fila (volumen sesgado): {avg_votes_post:.2f}")
print(f"Votos promedio por usuario (aisla volumen): {avg_votes_user:.2f}")

# Calcular la concentracion sobre el top 1% de usuarios
user_posts = df_core['user_id'].value_counts()
top_1_limit = max(1, int(len(user_posts) * 0.01))
top_1_total = user_posts.head(top_1_limit).sum()
top_1_pct = (top_1_total / total_core) * 100
print(f"\nEl {top_1_pct:.2f}% de los posts los genera el top 1% de usuarios.")

# Evaluar tendencias de tiempo por periodo mensual
print("\nActividad post por mes:")
print(df_core['post_created_at'].dt.to_period('M').value_counts().sort_index())

# Exportar core, cuarentena y resumen de metricas a CSV
df_core.to_csv('clean_product_activity.csv', index=False)
df_quarantine.to_csv('quarantine_product_activity.csv', index=False)
pd.DataFrame([{
    'core_rows': total_core,
    'avg_votes_per_row': avg_votes_post,
    'avg_votes_per_user': avg_votes_user,
    'quarantine_percent': quarantine_pct,
    'top_1_percent_post_share': top_1_pct
}]).to_csv('metrics_summary.csv', index=False)

print("\nResultados exportados exitosamente.")

### 4️⃣ Product Decisions
**Segmento a priorizar:** Retención del segmento Enterprise/Pro pues muestran actividad sostenible, de valor y mayor retención. Considerar el foco UI/UX para usuarios mobile que son una gran capa.
**Dato engañoso (antes de limpieza):** `days_since_signup` tenía graves desajustes (`mismatches`) al comparar con cálculos de fechas reales, lo que afectaba cualquier análisis de cohortes o ciclos de vida del usuario.
**Nuevo dato a trackear:** Tiempo promedio de permanencia leyendo el post o interacciones como comentarios/compartir, para que "votos recibidos" no sea la única métrica de validez.

**Acciones (basadas en datos):**
1. **Campañas de Activación (Retención):** El porcentaje de actividad está liderado excesivamente por el 1% de los usuarios. Se requiere implementar tácticas de gamificación / onboarding / recompensas a todos los usuarios regulares nuevos durante su primera semana para que se sientan motivados a publicar.
2. **Priorizar Desarrollo en App Web / Mobile:** `web` y `mobile` concentran abrumadoramente el uso frente a `desktop`. El equipo de ingeniería debería reenfocarse para que estas plataformas tengan paridad máxima en fluidez, y detener inversiones pesadas en `desktop`.